# 6. Data Cleaning & Transformation

This notebook implements the cleaning and transformation decisions established
during the Data Understanding / Data Audit stage.

Principles:

- Raw data remains unchanged.
- No cleaning is performed without an analytical reason.
- The main dataset preserves one row per unique Steam AppID.
- Multi-value dimensions are transformed without destroying the original game-level grain.
- Proxy variables are not interpreted as direct sales or revenue measures.

In [ ]:
import pandas as pd

file_path = "../data/raw/games.csv"

In [ ]:
# 원본 header만 먼저 읽기
original_columns = pd.read_csv(file_path, nrows=0).columns.tolist()

# 잘못 합쳐진 header 수정
corrected_columns = []

for col in original_columns:
    if col == "DiscountDLC count":
        corrected_columns.extend(["Discount", "DLC count"])
    else:
        corrected_columns.append(col)

# 수정된 40개 컬럼명으로 전체 데이터 로드
df = pd.read_csv(
    file_path,
    header=0,
    names=corrected_columns,
    index_col=False
)

In [ ]:
# Validation
print("Shape:", df.shape)
print("Number of columns:", len(df.columns))
print("Unique AppID:", df["AppID"].nunique())
print("Missing AppID:", df["AppID"].isna().sum())
print("Duplicated AppID:", df["AppID"].duplicated().sum())

### 6.1 Validation

The raw CSV contained a malformed header where `Discount` and `DLC count`
were merged into a single column name.

The header was corrected during import without modifying the raw file.

Validation confirmed:

- 125,855 rows
- 40 columns
- 125,855 unique AppIDs
- no missing AppIDs
- no duplicated AppIDs
- ---

## 6.2 Column Selection

Columns identified during the audit as having no meaningful role in the current business questions are removed from the working dataset.

Secondary variables are retained at this stage so that potentially useful information is not discarded prematurely.

In [ ]:
# Audit 단계에서 분석 대상에서 제외하기로 결정한 컬럼
columns_to_drop = [
    "About the game",
    "Full audio languages",
    "Reviews",
    "Header image",
    "Website",
    "Support url",
    "Support email",
    "Metacritic url",
    "Score rank",
    "Notes",
    "Screenshots",
    "Movies"
]

# 원본 표현 df는 유지하고, cleaning용 DataFrame을 별도로 생성
df_clean = df.drop(columns=columns_to_drop).copy()

In [ ]:
# 컬럼 제거 전후 크기 확인
print("Before column selection:", df.shape)
print("After column selection:", df_clean.shape)

# 제거하기로 한 컬럼이 실제로 모두 사라졌는지 확인
remaining_dropped_columns = [
    col for col in columns_to_drop
    if col in df_clean.columns
]

print("Dropped columns still present:", remaining_dropped_columns)

In [ ]:
# 현재 분석 후보 컬럼 확인
df_clean.columns.tolist()

### 6.2 Validation

12 columns identified as outside the current analytical scope were removed.

The working cleaning dataset now contains:

- 125,855 records
- 28 candidate analytical columns

Secondary variables were intentionally retained for later transformation or
analysis-specific use.

---
## 6.3 Record & Product Scope Cleaning

The audit identified several records that should not automatically be treated
as standard released Steam games.

This step:

- removes one analytically unusable record,
- identifies Playtest applications,
- flags software-oriented genre records without automatically excluding them.

Product-scope decisions are applied conservatively to avoid removing valid
games based only on unusual genre labels.

In [ ]:
# 분석 정보가 거의 존재하지 않았던 unusable record 제거
df_clean = df_clean[
    df_clean["AppID"] != 396420
].copy()

# 제거 결과 확인
print("Rows after unusable record removal:", len(df_clean))
print(
    "AppID 396420 remaining:",
    (df_clean["AppID"] == 396420).sum()
)

In [ ]:
# 이름에 "Playtest"가 포함된 Steam application 식별
df_clean["is_playtest"] = (
    df_clean["Name"]
    .fillna("")
    .str.contains("Playtest", case=False, regex=False)
)

# Playtest 개수 확인
print("Playtest records:", df_clean["is_playtest"].sum())

In [ ]:
# Playtest를 제외한 main market dataset 생성
df_market = df_clean[
    ~df_clean["is_playtest"]
].copy()

# 전후 row 수 확인
print("Before Playtest exclusion:", len(df_clean))
print("After Playtest exclusion:", len(df_market))
print("Excluded Playtests:", df_clean["is_playtest"].sum())

In [ ]:
# Audit에서 확인한 software-oriented genre labels
software_like_genres = [
    "Animation & Modeling",
    "Design & Illustration",
    "Utilities",
    "Video Production",
    "Audio Production",
    "Web Publishing",
    "Software Training",
    "Game Development",
    "Photo Editing"
]

In [ ]:
# 쉼표로 구분된 Genres를 임시로 분리한 뒤
# software-oriented genre가 하나라도 포함되어 있는지 확인
df_market["has_software_genre"] = (
    df_market["Genres"]
    .fillna("")
    .str.split(",")
    .apply(
        lambda genres:
        any(
            genre.strip() in software_like_genres
            for genre in genres
        )
    )
)

# 해당 레코드 수 확인
print(
    "Records with software-oriented genres:",
    df_market["has_software_genre"].sum()
)

In [ ]:
# software-oriented genre가 포함된 실제 사례 확인
df_market.loc[
    df_market["has_software_genre"],
    [
        "AppID",
        "Name",
        "Genres",
        "Price"
    ]
].head(20)

In [ ]:
# main market dataset의 key integrity 확인
print("Market dataset shape:", df_market.shape)
print("Unique AppID:", df_market["AppID"].nunique())
print("Missing AppID:", df_market["AppID"].isna().sum())
print("Duplicated AppID:", df_market["AppID"].duplicated().sum())
print("Playtests remaining:", df_market["is_playtest"].sum())

### 6.3 Validation

One analytically unusable record (`AppID 396420`) was removed.

Playtest applications were identified using the application name and retained in the broader cleaned dataset through an `is_playtest` flag.

A separate main market dataset (`df_market`) was then created excluding 8,139 Playtest records.

Software-oriented genre labels were not used as automatic exclusion criteria.
Instead, these records were flagged with `has_software_genre` because some applications combine software-related labels with conventional game genres.

The main market dataset continues to preserve one row per unique Steam AppID.

---
## 6.4 Data Type Transformation

Data types are converted only where required for analysis.

`Release date` was successfully parsed for all records during the audit.
It is therefore converted from string to datetime in the cleaned market dataset.

A `release_year` variable is derived for supply and market-trend analysis.
The original release date remains available for more detailed time-based analysis.

In [ ]:
# Release date를 문자열(str)에서 실제 datetime 타입으로 변환
# Audit에서 모든 값이 정상적으로 parse되는 것을 확인했으므로
# 여기서는 문제가 발생하면 바로 알 수 있도록 errors="raise" 사용
df_market["Release date"] = pd.to_datetime(
    df_market["Release date"],
    errors="raise"
)

# Market Trend 분석(Q5)을 위한 출시 연도 파생변수 생성
df_market["release_year"] = df_market["Release date"].dt.year

In [ ]:
# 변환된 dtype 확인
print("Release date dtype:", df_market["Release date"].dtype)
print("Release year dtype:", df_market["release_year"].dtype)

# 변환 과정에서 결측값이 발생하지 않았는지 확인
print("Missing release dates:", df_market["Release date"].isna().sum())
print("Missing release years:", df_market["release_year"].isna().sum())

# 연도 범위 확인
print("Earliest release year:", df_market["release_year"].min())
print("Latest release year:", df_market["release_year"].max())

In [ ]:
# 프로젝트의 데이터 기준일
analysis_date = pd.Timestamp("2026-09-22")

# 분석 기준일보다 이후에 출시되는 application 표시
df_market["is_future_release"] = (
    df_market["Release date"] > analysis_date
)

# 미래 출시 레코드 수 확인
print(
    "Future releases:",
    df_market["is_future_release"].sum()
)

In [ ]:
# 주요 분석 변수들의 현재 dtype 확인
dtype_check_columns = [
    "AppID",
    "Peak CCU",
    "Price",
    "Discount",
    "DLC count",
    "Positive",
    "Negative",
    "Recommendations",
    "Windows",
    "Mac",
    "Linux",
    "Release date",
    "release_year",
    "is_playtest",
    "has_software_genre",
    "is_future_release"
]

df_market[dtype_check_columns].dtypes

### 6.4 Validation

`Release date` was successfully converted to datetime without introducing missing values.

`release_year` was derived for annual supply and trend analysis.

Future releases were retained and flagged rather than globally removed,allowing analysis-specific filtering later.

Existing numeric and boolean variables were preserved unless a transformation
was analytically required.

---
## 6.5 Categorical Transformation

`Genres`, `Categories`, and `Tags` contain multiple values within a single
string field.

To preserve the main dataset grain (`1 row = 1 unique AppID`), these variables
are not exploded inside `df_market`.

Instead, separate AppID-to-category mapping tables are created for later
segmentation and many-to-many analysis.

Developers and Publishers are not split because commas may occur inside entity
names. Supported languages are also retained in their original form because
localization analysis is outside the current core scope.

In [ ]:
# comma-separated multi-value 컬럼을
# AppID ↔ 개별 category 형태의 mapping table로 변환하는 함수
def create_mapping_table(dataframe, source_column, output_column):
    
    # AppID와 대상 컬럼만 가져오고 결측값 제거
    mapping = dataframe[
        ["AppID", source_column]
    ].dropna().copy()
    
    # "Action,Indie,RPG" → ["Action", "Indie", "RPG"]
    mapping[source_column] = (
        mapping[source_column]
        .str.split(",")
    )
    
    # 리스트의 각 값을 별도 row로 펼침
    mapping = mapping.explode(source_column)
    
    # 앞뒤 공백 제거
    mapping[source_column] = (
        mapping[source_column]
        .str.strip()
    )
    
    # 혹시 존재할 수 있는 빈 문자열 제거
    mapping = mapping[
        mapping[source_column] != ""
    ]
    
    # 분석하기 쉬운 이름으로 컬럼명 변경
    mapping = mapping.rename(
        columns={source_column: output_column}
    )
    
    # 동일 AppID에 같은 값이 중복되어 있을 경우 제거
    mapping = mapping.drop_duplicates(
        subset=["AppID", output_column]
    )
    
    # index 정리
    mapping = mapping.reset_index(drop=True)
    
    return mapping

In [ ]:
# AppID ↔ Genre mapping
game_genres = create_mapping_table(
    df_market,
    "Genres",
    "Genre"
)

# AppID ↔ Category mapping
game_categories = create_mapping_table(
    df_market,
    "Categories",
    "Category"
)

# AppID ↔ Tag mapping
game_tags = create_mapping_table(
    df_market,
    "Tags",
    "Tag"
)

In [ ]:
game_genres.head(10)

In [ ]:
game_categories.head(10)

In [ ]:
game_tags.head(10)

In [ ]:
# 각 mapping table의 기본 구조 확인
print(
    "Genres:",
    len(game_genres), "rows |",
    game_genres["AppID"].nunique(), "AppIDs |",
    game_genres["Genre"].nunique(), "unique genres"
)

print(
    "Categories:",
    len(game_categories), "rows |",
    game_categories["AppID"].nunique(), "AppIDs |",
    game_categories["Category"].nunique(), "unique categories"
)

print(
    "Tags:",
    len(game_tags), "rows |",
    game_tags["AppID"].nunique(), "AppIDs |",
    game_tags["Tag"].nunique(), "unique tags"
)

In [ ]:
# 동일 AppID에 동일 category가 중복되어 있지 않은지 확인
print(
    "Duplicate genre pairs:",
    game_genres.duplicated(
        subset=["AppID", "Genre"]
    ).sum()
)

print(
    "Duplicate category pairs:",
    game_categories.duplicated(
        subset=["AppID", "Category"]
    ).sum()
)

print(
    "Duplicate tag pairs:",
    game_tags.duplicated(
        subset=["AppID", "Tag"]
    ).sum()
)

In [ ]:
# main game-level dataset의 grain이 유지되는지 확인
print("Market rows:", len(df_market))
print("Unique AppIDs:", df_market["AppID"].nunique())
print(
    "Duplicated AppIDs:",
    df_market["AppID"].duplicated().sum()
)

### 6.5 Validation

Separate mapping tables were created for Genres, Categories, and Tags while
preserving the original one-row-per-AppID structure of `df_market`.

The resulting tables are:

- `game_genres`: AppID ↔ Genre
- `game_categories`: AppID ↔ Category
- `game_tags`: AppID ↔ Tag

Duplicate AppID-category pairs were removed.

These mapping tables support many-to-many analysis without duplicating game-level
metrics inside the main dataset.

A game may belong to multiple genres, categories, or tags. Therefore, totals
across these groups are not necessarily additive: the same AppID may legitimately
contribute to more than one group.

---
## 6.6 Estimated Owners Transformation

`Estimated owners` is provided as a set of non-uniform ownership ranges rather
than exact numeric counts.

The original range is preserved.

For analysis:

- lower and upper bounds are extracted,
- `0 - 0` is flagged as an uncertain ownership estimate rather than interpreted
  as confirmed zero ownership,
- an ordered ownership-band variable is created for relative traction comparison.

No midpoint estimate is created because the ranges have unequal widths and a
midpoint would imply greater precision than the source data provides.

In [ ]:
# "0 - 0"은 실제 owner가 정확히 0명이라고 볼 수 없으므로
# 별도의 uncertainty flag로 표시
df_market["owner_estimate_uncertain"] = (
    df_market["Estimated owners"] == "0 - 0"
)

# 개수 확인
print(
    "Uncertain owner estimates:",
    df_market["owner_estimate_uncertain"].sum()
)

In [ ]:
# Estimated owners 범위를 lower / upper로 분리
owner_bounds = (
    df_market["Estimated owners"]
    .str.split("-", expand=True)
)

# 공백 제거 후 숫자로 변환
owner_lower = (
    owner_bounds[0]
    .str.strip()
    .astype("Int64")
)

owner_upper = (
    owner_bounds[1]
    .str.strip()
    .astype("Int64")
)

In [ ]:
# lower bound가 upper bound보다 큰 잘못된 범위가 있는지 확인
print(
    "Invalid owner ranges:",
    (owner_lower > owner_upper).sum()
)

In [ ]:
# uncertainty 상태인 "0 - 0"에는 숫자형 bound를 부여하지 않음
owner_lower = owner_lower.mask(
    df_market["owner_estimate_uncertain"]
)

owner_upper = owner_upper.mask(
    df_market["owner_estimate_uncertain"]
)

# 분석용 bound 컬럼 추가
df_market["owner_lower"] = owner_lower
df_market["owner_upper"] = owner_upper

In [ ]:
# "0 - 0"을 제외한 실제 owner range들의 순서를 자동 생성
owner_band_order = (
    df_market.loc[
        ~df_market["owner_estimate_uncertain"],
        ["Estimated owners", "owner_lower", "owner_upper"]
    ]
    .drop_duplicates()
    .sort_values(["owner_lower", "owner_upper"])
    ["Estimated owners"]
    .tolist()
)

owner_band_order

In [ ]:
# ownership range를 순서가 있는 categorical 변수로 생성
# "0 - 0"은 uncertain이므로 owner_band에서는 결측으로 처리
df_market["owner_band"] = pd.Categorical(
    df_market["Estimated owners"].where(
        ~df_market["owner_estimate_uncertain"]
    ),
    categories=owner_band_order,
    ordered=True
)

In [ ]:
# 생성된 owner 관련 변수 확인
df_market[
    [
        "Estimated owners",
        "owner_estimate_uncertain",
        "owner_lower",
        "owner_upper",
        "owner_band"
    ]
].head(20)

In [ ]:
# 변환 결과 검증
print(
    "Uncertain estimates:",
    df_market["owner_estimate_uncertain"].sum()
)

print(
    "Missing owner_lower:",
    df_market["owner_lower"].isna().sum()
)

print(
    "Missing owner_upper:",
    df_market["owner_upper"].isna().sum()
)

print(
    "Missing owner_band:",
    df_market["owner_band"].isna().sum()
)

print(
    "Invalid numeric ranges:",
    (
        df_market["owner_lower"] >
        df_market["owner_upper"]
    ).sum()
)

In [ ]:
# band가 제대로 ordered categorical인지
print("owner_band dtype:", df_market["owner_band"].dtype)

print("\nOwner band order:")
for band in df_market["owner_band"].cat.categories:
    print(band)

### 6.6 Validation

The original `Estimated owners` range was preserved.

Three analytical representations were added:

- `owner_estimate_uncertain`: identifies `0 - 0` records whose ownership cannot
  safely be interpreted as confirmed zero,
- `owner_lower` and `owner_upper`: numeric boundaries for valid ownership ranges,
- `owner_band`: an ordered categorical representation of the ownership ranges.

For uncertain `0 - 0` records, numeric bounds and owner band values are left
missing to prevent accidental interpretation as zero ownership.

No midpoint estimate was created.

---
## 6.7 Analytical Derived Metrics

Several analysis-ready metrics are derived from the cleaned source variables.

Derived metrics are created only where their analytical meaning is clear.

The following variables are added:

- `review_volume`: total observed positive and negative review counts,
- `has_reviews`: whether any review signal is available,
- `positive_ratio`: share of positive reviews among observed positive and
  negative reviews,
- `is_free`: whether the listed price is zero.

These variables are analytical proxies and should not be interpreted as sales,
revenue, or complete measures of player satisfaction.

In [ ]:
# Positive + Negative를 이용해 관측된 review activity volume 생성
df_market["review_volume"] = (
    df_market["Positive"] +
    df_market["Negative"]
)

In [ ]:
# Positive 또는 Negative review가 하나라도 존재하는지 표시
df_market["has_reviews"] = (
    df_market["review_volume"] > 0
)

In [ ]:
# review가 존재하는 경우에만 positive review 비율 계산
# review_volume == 0인 경우에는 satisfaction을 측정할 수 없으므로 NA 처리
df_market["positive_ratio"] = (
    df_market["Positive"] /
    df_market["review_volume"]
).where(
    df_market["has_reviews"]
)

In [ ]:
# 현재 listed price가 0인 application을 free로 표시
df_market["is_free"] = (
    df_market["Price"] == 0
)

In [ ]:
# Derived metric 기본 검증
print(
    "Games with review signal:",
    df_market["has_reviews"].sum()
)

print(
    "Missing positive_ratio:",
    df_market["positive_ratio"].isna().sum()
)

print(
    "positive_ratio below 0:",
    (df_market["positive_ratio"] < 0).sum()
)

print(
    "positive_ratio above 1:",
    (df_market["positive_ratio"] > 1).sum()
)

print(
    "Free games:",
    df_market["is_free"].sum()
)

In [ ]:
print(
    "Missing positive_ratio:",
    df_market["positive_ratio"].isna().sum()
)

print(
    "Rows with no review signal:",
    (df_market["review_volume"] == 0).sum()
)

print(
    "Rows where has_reviews is False:",
    (~df_market["has_reviews"]).sum()
)

In [ ]:
# positive_ratio의 결측값은 review_volume == 0인 행과 같아야 함
print(
    "Missing positive_ratio:",
    df_market["positive_ratio"].isna().sum()
)

print(
    "Rows with no review signal:",
    (df_market["review_volume"] == 0).sum()
)

# positive_ratio는 0~1 범위를 벗어나면 안 됨
print(
    "positive_ratio below 0:",
    (df_market["positive_ratio"] < 0).sum()
)

print(
    "positive_ratio above 1:",
    (df_market["positive_ratio"] > 1).sum()
)

# free flag와 price == 0 관계 확인
print(
    "Free games:",
    df_market["is_free"].sum()
)

print(
    "Price == 0:",
    (df_market["Price"] == 0).sum()
)

### 6.7 Validation

Several analysis-ready metrics were derived from the cleaned source variables.

- `review_volume` represents the total observed positive and negative review counts.
- `has_reviews` identifies records with at least one observed review.
- `positive_ratio` is calculated only when review activity exists.
  Records with `review_volume = 0` remain missing rather than being interpreted
  as zero satisfaction.
- `is_free` identifies applications whose listed price is zero.

Validation confirmed that:

- missing `positive_ratio` values correspond only to records without review activity,
- all valid `positive_ratio` values fall between 0 and 1,
- the `is_free` flag is consistent with `Price = 0`.

These derived variables are analytical proxies and are not interpreted as
sales, revenue, or complete measures of player satisfaction.

---
## 6.8 Final Validation

Before saving the processed datasets, the cleaned market-level dataset and
categorical mapping tables are validated for structural integrity and logical
consistency.

This step does not introduce additional cleaning rules.
Its purpose is to confirm that the transformations implemented in this notebook
did not create duplicated keys, invalid ranges, or inconsistent derived metrics.

In [ ]:
# ---------------------------------------
# 1. Main dataset grain validation
# ---------------------------------------

print("Rows:", len(df_market))
print("Unique AppIDs:", df_market["AppID"].nunique())
print("Missing AppIDs:", df_market["AppID"].isna().sum())
print("Duplicated AppIDs:", df_market["AppID"].duplicated().sum())

In [ ]:
# ---------------------------------------
# 2. Product scope validation
# ---------------------------------------

print(
    "Playtests remaining:",
    df_market["is_playtest"].sum()
)

print(
    "Future releases:",
    df_market["is_future_release"].sum()
)

print(
    "Software-genre flagged records:",
    df_market["has_software_genre"].sum()
)

In [ ]:
# ---------------------------------------
# 3. Estimated owners validation
# ---------------------------------------

print(
    "Uncertain owner estimates:",
    df_market["owner_estimate_uncertain"].sum()
)

print(
    "Missing owner_lower:",
    df_market["owner_lower"].isna().sum()
)

print(
    "Missing owner_upper:",
    df_market["owner_upper"].isna().sum()
)

print(
    "Missing owner_band:",
    df_market["owner_band"].isna().sum()
)

print(
    "Invalid owner ranges:",
    (
        df_market["owner_lower"] >
        df_market["owner_upper"]
    ).sum()
)

In [ ]:
# uncertain이 아닌데 owner bound가 없는 경우 확인
unexpected_owner_missing = (
    ~df_market["owner_estimate_uncertain"]
    &
    (
        df_market["owner_lower"].isna()
        |
        df_market["owner_upper"].isna()
    )
)

print(
    "Unexpected missing owner bounds:",
    unexpected_owner_missing.sum()
)

In [ ]:
# ---------------------------------------
# 4. Derived metric validation
# ---------------------------------------

print(
    "Negative review_volume:",
    (df_market["review_volume"] < 0).sum()
)

print(
    "Positive ratio below 0:",
    (df_market["positive_ratio"] < 0).sum()
)

print(
    "Positive ratio above 1:",
    (df_market["positive_ratio"] > 1).sum()
)

print(
    "No-review rows:",
    (~df_market["has_reviews"]).sum()
)

print(
    "Missing positive_ratio:",
    df_market["positive_ratio"].isna().sum()
)

print(
    "is_free mismatch:",
    (
        df_market["is_free"]
        !=
        (df_market["Price"] == 0)
    ).sum()
)

In [ ]:
# ---------------------------------------
# 5. Mapping table validation
# ---------------------------------------

print(
    "Duplicate genre pairs:",
    game_genres.duplicated(
        subset=["AppID", "Genre"]
    ).sum()
)

print(
    "Duplicate category pairs:",
    game_categories.duplicated(
        subset=["AppID", "Category"]
    ).sum()
)

print(
    "Duplicate tag pairs:",
    game_tags.duplicated(
        subset=["AppID", "Tag"]
    ).sum()
)

In [ ]:
# ---------------------------------------
# 6. Mapping table foreign-key validation
# ---------------------------------------

market_appids = set(df_market["AppID"])

print(
    "Genre AppIDs not in df_market:",
    (~game_genres["AppID"].isin(market_appids)).sum()
)

print(
    "Category AppIDs not in df_market:",
    (~game_categories["AppID"].isin(market_appids)).sum()
)

print(
    "Tag AppIDs not in df_market:",
    (~game_tags["AppID"].isin(market_appids)).sum()
)

In [ ]:
# ---------------------------------------
# 7. Final dataset overview
# ---------------------------------------

print("df_market:", df_market.shape)
print("game_genres:", game_genres.shape)
print("game_categories:", game_categories.shape)
print("game_tags:", game_tags.shape)

### 6.8 Validation Summary

Final validation confirmed that the processed market dataset preserves the
intended game-level grain:

- one row per unique AppID,
- no missing or duplicated AppIDs,
- no Playtest applications in the main market dataset.

Estimated-owner transformations were internally consistent, with uncertain
`0 - 0` estimates kept separate from numeric owner bounds.

Derived review and pricing metrics passed logical range and consistency checks.

Genre, Category, and Tag mapping tables contain no duplicated AppID-value pairs
and reference only AppIDs that exist in the main market dataset.

The cleaned datasets are therefore ready to be saved as processed analytical
data.

---
## 6.9 Save Processed Data

The validated analytical datasets are saved separately from the raw source data.

The processed data structure preserves:

- one row per unique AppID in the main market dataset,
- separate many-to-many mapping tables for Genres, Categories, and Tags,
- all cleaning and transformation decisions established during the audit.

The original raw CSV remains unchanged.

In [ ]:
from pathlib import Path

# processed 데이터 저장 폴더 경로
processed_dir = Path("../data/processed")

# 폴더가 없으면 생성
processed_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# ---------------------------------------
# 1. Main cleaned market dataset 저장
# ---------------------------------------

df_market.to_csv(
    processed_dir / "games_clean.csv",
    index=False,
    date_format="%Y-%m-%d"
)

In [ ]:
# ---------------------------------------
# 2. Genre mapping table 저장
# ---------------------------------------

game_genres.to_csv(
    processed_dir / "game_genres.csv",
    index=False
)

# ---------------------------------------
# 3. Category mapping table 저장
# ---------------------------------------

game_categories.to_csv(
    processed_dir / "game_categories.csv",
    index=False
)

# ---------------------------------------
# 4. Tag mapping table 저장
# ---------------------------------------

game_tags.to_csv(
    processed_dir / "game_tags.csv",
    index=False
)

In [ ]:
# 저장된 파일 확인
for file in processed_dir.iterdir():
    print(file.name)

In [ ]:
# ---------------------------------------
# 5. 저장된 main dataset 재로드
# ---------------------------------------

df_saved = pd.read_csv(
    processed_dir / "games_clean.csv"
)

print("Saved shape:", df_saved.shape)
print("Original shape:", df_market.shape)

print(
    "Saved unique AppIDs:",
    df_saved["AppID"].nunique()
)

print(
    "Saved duplicated AppIDs:",
    df_saved["AppID"].duplicated().sum()
)

In [ ]:
# ---------------------------------------
# 6. Mapping table 저장 결과 검증
# ---------------------------------------

genres_saved = pd.read_csv(
    processed_dir / "game_genres.csv"
)

categories_saved = pd.read_csv(
    processed_dir / "game_categories.csv"
)

tags_saved = pd.read_csv(
    processed_dir / "game_tags.csv"
)

print(
    "Genres:",
    len(game_genres),
    "→",
    len(genres_saved)
)

print(
    "Categories:",
    len(game_categories),
    "→",
    len(categories_saved)
)

print(
    "Tags:",
    len(game_tags),
    "→",
    len(tags_saved)
)

### 6.9 Final Output

The cleaning and transformation pipeline produced four processed datasets:

- `games_clean.csv` — main game-level analytical dataset
- `game_genres.csv` — AppID-to-Genre mapping
- `game_categories.csv` — AppID-to-Category mapping
- `game_tags.csv` — AppID-to-Tag mapping

All processed files were validated after export.

The raw source file remains unchanged.

The resulting data structure is ready for exploratory analysis, PostgreSQL
loading, and later Power BI reporting.